# SD3.5 CityPersons Augmentation Runner

Runner notebook gon cho Kaggle. Notebook nay clone repo tu GitHub vao `/kaggle/working/VIN`, sau do import truc tiep cac module `sd35_*.py` tu repo da clone.

## 1. Install Dependencies

In [ ]:
!pip install -q "diffusers>=0.30.0,<1.0.0" "transformers>=4.40.0" "accelerate>=0.30.0" sentencepiece protobuf safetensors ultralytics opencv-python


## 2. Clone Or Update Repo

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/BDT-17/VIN.git'
REPO_DIR = Path('/kaggle/working/VIN')

if REPO_DIR.exists():
    %cd /kaggle/working/VIN
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /kaggle/working/VIN

PROJECT_DIR = REPO_DIR / 'notebooks'
if not (PROJECT_DIR / 'sd35_config.py').exists():
    PROJECT_DIR = REPO_DIR

%cd {PROJECT_DIR}
print('PROJECT_DIR:', PROJECT_DIR)


## 3. Imports

In [ ]:

import sys
import importlib
from pathlib import Path

module_dirs = [str(PROJECT_DIR), str(Path('/kaggle/working'))]
sys.path = module_dirs + [path for path in sys.path if path not in module_dirs]

for module_name in list(sys.modules):
    if module_name.startswith('sd35_'):
        del sys.modules[module_name]

from sd35_config import *
from sd35_data import *
from sd35_utils import *
from sd35_model import *
from sd35_evaluation import *
from sd35_pipeline import *
from sd35_runner import *
import sd35_runner

ensure_output_dirs()
print('sd35_runner:', sd35_runner.__file__)
print('generation pipeline:', CONTEXT_PERSON_GENERATION_PIPELINE)
print('output dir:', OUTPUT_DIR)


## 4. Runtime Check

In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu count:', torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        print(index, torch.cuda.get_device_name(index))


## 5. Hugging Face Login

In [ ]:
from huggingface_hub import login

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    import os
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("HF_TOKEN not found. Add a Kaggle secret named HF_TOKEN or set os.environ['HF_TOKEN'].") from exc

login(token=hf_token)
print("Hugging Face login OK from HF_TOKEN secret.")


## 6. Dataset Scan

In [ ]:
records = load_records()
summarize_citypersons_records(records)
preview_prompt_samples(records)


## 7. Smoke Run

In [ ]:
SMOKE_IMAGES = 10
SMOKE_SPLITS = ['train']

generated_paths, manifest_rows, autotune_report = run_smoke(
    records,
    smoke_images=SMOKE_IMAGES,
    smoke_splits=SMOKE_SPLITS,
)

generated_paths[:5]


## 8. Export Outputs

In [ ]:
# Run after generation if you want a zip artifact.
export_outputs()
